# 病人 MSN 构建与拓扑提取

本 notebook 针对 **眩晕患者（patient 组）**：

- 从 `data/patient_feature_matrices/` 读取每个被试的特征矩阵（行=脑区/体积指标，列=特征）。
- 使用 `msn_pipeline.msn.build_msn` 为每个被试构建 MSN（参数与主 `run_msn_pipeline` 完全一致，例如 `metric="pearson"`, `target_density=0.2`）。
- 使用 `msn_pipeline.features.calculate_graph_metrics` 提取全局拓扑指标（Eg, Eloc, Cp, Lp，σ 可选）。
- 使用 `msn_pipeline.features.compute_nodal_topology` 提取节点级拓扑指标（degree_centrality、betweenness、clustering、eigenvector、pagerank）。
- 将结果导出为：
  - 每个患者的 MSN 矩阵 CSV（**Pearson 相关经 Fisher Z 变换后保存**），目录为 `results/patient_msn_matrices/`，文件名约定为 `{subject_id}_msn.csv`。
  - 汇总的全局拓扑指标表 `results/patient_subject_topology.csv`（包含 `subject_id`, `group`=1 以及各全局拓扑指标列）。
  - （可选）每个患者的节点级拓扑指标表，目录为 `results/patient_nodal_topology/`（供后续 GLM/ANCOVA 使用）。

> 本 notebook 的目标是为后续 **健康 vs 患者** 以及 **患者亚型间** 的拓扑/NBS/GLM 分析提供干净一致的患者 MSN 及拓扑特征表。

# 1.环境和路径


In [ ]:
import sys
from pathlib import Path

# 自动识别项目根：当前目录或上一级（在 notebook/ 下运行时）
ROOT = Path.cwd().resolve()
if not (ROOT / "src" / "msn_pipeline").exists():
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

print("项目根:", ROOT)
print("msn_pipeline 可导入:", (ROOT / "src" / "msn_pipeline").exists())

# 2. 参数配置

集中设置 MSN 构建与拓扑提取所需的关键参数与路径，保证：

- **与主管线一致**：`metric`、`target_density` 等必须与 `run_msn_pipeline` 中使用的参数一致。
- **输出结构统一**：所有患者 MSN 矩阵与拓扑结果写入 `results/` 下固定的子目录，方便后续 NBS 与 GLM 统一读取。
- **可选开关清晰**：
  - 是否计算小世界系数 σ（可能较慢）。
  - 是否为每个患者保存节点级拓扑指标 CSV。

In [ ]:
import pandas as pd
import numpy as np

from msn_pipeline.data import DataPaths, load_subject_features
from msn_pipeline.msn import build_msn
from msn_pipeline.features import calculate_graph_metrics, compute_nodal_topology
from msn_pipeline.pipelines import build_group_msns

# 基础路径（与项目结构保持一致）
DATA_ROOT = ROOT / "data"
RESULTS_ROOT = ROOT / "results"

# MSN 参数，应与主管线保持一致
MSN_METRIC = "pearson"
TARGET_DENSITY = 0.2

# 是否计算小世界 σ（较慢），可按需打开
COMPUTE_SIGMA = False
# 是否为每个被试保存节点级拓扑指标
SAVE_NODAL = False

# 患者 MSN / 节点拓扑输出目录（与顶部 markdown 约定一致）
PATIENT_MSN_DIR = RESULTS_ROOT / "patient_msn_matrices"
PATIENT_NODAL_DIR = RESULTS_ROOT / "patient_nodal_topology"
PATIENT_MSN_DIR.mkdir(parents=True, exist_ok=True)
PATIENT_NODAL_DIR.mkdir(parents=True, exist_ok=True)


# 3. 加载患者特征矩阵

从 `data/patient_feature_matrices/` 加载每个患者的特征矩阵，检查被试数量与单个被试表格的基本形状，确保：

- 行对应脑区/体积指标，与 MSN 中节点一一对应。
- 列对应不同的影像/临床特征，将作为构建 MSN 的节点特征。

In [ ]:
# 患者特征矩阵目录由 DataPaths 统一管理
paths = DataPaths(root=DATA_ROOT)
patient_dir = paths.patient_csv_dir()
print("患者特征矩阵目录:", patient_dir)

subjects = load_subject_features(patient_dir)
print("加载患者数量:", len(subjects))
if subjects:
    sid, df = next(iter(subjects.items()))
    print("示例 subject_id:", sid, "特征表形状:", df.shape)

# 4. 为每个患者构建 MSN 并保存

本节对每个患者：

- 使用 `build_msn` 构建个体 MSN 网络（相似度方法为 **Pearson 相关**，与主管线一致）。
- 对得到的 Pearson 相关矩阵做 **Fisher Z 变换**（z = arctanh(r)），再保存为 CSV，便于后续组间/NBS 等统计（Z 近似正态、方差稳定）。
- 将 Fisher Z 矩阵保存到 `results/patient_msn_matrices/`（行列均为脑区名称）。
- 同时计算全局拓扑指标，并（可选）计算/保存节点级拓扑指标，供下一节汇总为特征表。

In [ ]:
# 为每个患者构建 MSN、保存矩阵，并记录全局/节点拓扑信息
# 重构后：调用 msn_pipeline.pipelines.build_group_msns 承担核心逻辑

patient_topology_df = build_group_msns(
    group="patient",
    data_root=DATA_ROOT,
    results_root=RESULTS_ROOT,
    metric=MSN_METRIC,
    target_density=TARGET_DENSITY,
    compute_sigma=COMPUTE_SIGMA,
    save_nodal=SAVE_NODAL,
)

print("build_msn 完成，患者数量:", len(patient_topology_df))
if not patient_topology_df.empty:
    print("示例 subject_id:", patient_topology_df["subject_id"].iloc[0])



# 5. 汇总全局拓扑指标并导出

本节将上一节循环中得到的每个患者的全局拓扑指标汇总成 `DataFrame`，并：

- 添加 `group` 列，用于区分健康/患者（约定患者 `group=1`）。
- 统一列顺序为：`subject_id`, `group`, 各拓扑指标列。
- 输出为 `results/patient_subject_topology.csv`，供后续 NBS、GLM/ANCOVA 分析使用。

In [ ]:
# 将 global_rows 汇总为 DataFrame，并保存为患者全局拓扑主表

patient_topology_df = pd.DataFrame(global_rows)
if patient_topology_df.empty:
    raise RuntimeError("global_rows 为空，检查上一节是否成功运行并收集到拓扑指标。")

# 添加 group 列：患者记为 1
patient_topology_df["group"] = 1

# 统一列顺序：subject_id, group, 其余拓扑指标列
cols = ["subject_id", "group"] + [
    c for c in patient_topology_df.columns
    if c not in ("subject_id", "group")
]
patient_topology_df = patient_topology_df[cols]

# 保存到 results/patient_subject_topology.csv
topology_path = RESULTS_ROOT / "patient_subject_topology.csv"
topology_path.parent.mkdir(parents=True, exist_ok=True)
patient_topology_df.to_csv(topology_path, index=False)
print("已保存患者全局拓扑指标到:", topology_path)
patient_topology_df.head()